# LAB Patient Features

This notebook generates patient-level features for **Lab test claims** using the Generic Patient Feature Engineering methodology.

## Features Generated

1. **First Occurrence** -- Days before anchor when each lab test first appeared (full lookback)
2. **Recency** -- How recently/frequently events occurred (12M lookback, 6 sub-features)
3. **Frequency** -- Monthly claim counts, rolling lags, change in frequency, unique patient counts
4. **Hospitalization** -- NOT applicable for Lab (DX/PX only)

## Source Mapping (Lab-specific)

| Canonical | Lab Raw |
|---|---|
| `EVENT_NAME` | `TEST_TYPE` |
| `EVENT_TYPE` | `lab` |
| `EVENT_DATE` | `SVC_DT` |

## Lab-specific column naming for Freq Sub-D
`NUM_OF_PATIENTS_THE_HCP_DID_LAB_TEST_{EVENT}_IN_LAST_{N}_DAYS`

## Lab-specific: Typical event count ~2 distinct test types

In [0]:
# ============================================================================
# CONFIG PARAMETERS (from 00_config)
# ============================================================================

cohort_start_date = "2026-06-01"
cohort_end_date = "2026-06-01"
months_increment = 2
lookback_duration = 365
prediction_duration = 90
cohort_day_offset = 1

# Sampling parameters
sample_hcp_count = 10000                 # Randomly sample N HCPs (0 = use all HCPs)

event_prevalence = 0.00
batch_size = 6

time_windows_recency = [30, 60, 90, 180, 270, 360]
lag_windows_freq = [1, 2, 3, 6, 12]
change_windows_freq = [1, 2, 3, 6]
patient_count_windows_freq = [30, 60, 90, 180, 360]

# File paths
lab_file_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/lab_input_claims_12M_and_first_occ_features_input"
output_base_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
anchor_dates_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/hcp_model_inference_anchor_dates.csv"

print("Config parameters loaded.")

In [0]:
# ============================================================================
# IMPORTS AND STANDARDIZATION FUNCTION
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
from datetime import datetime, timedelta

COLUMN_MAP = {
    "PROVIDER_ID": ["PROVIDER_ID", "HCP_ID", "PHYSICIAN_ID", "PROV_ID", "BHID"],
    "PATIENT_ID":  ["PATIENT_ID", "MEMBER_ID", "ENROLLEE_ID", "SUBSCRIBER_ID"],
    "EVENT_NAME":  ["PRODUCT_GROUP", "DIAGNOSIS_DESCRIPTION", "PROCEDURE_DESCRIPTION", "TEST_TYPE"],
    "EVENT_DATE":  ["SVC_DT", "SERVICE_DATE", "CLAIM_DATE", "FILL_DT"],
    "CLAIM_ID":    ["CLAIM_ID", "CLAIM_NBR", "CLAIM_NUMBER"],
}

def standardize(df, event_type):
    """Standardize raw claims to canonical schema (Section 1.3)."""
    for canonical, variants in COLUMN_MAP.items():
        for v in variants:
            if v in df.columns and canonical not in df.columns:
                df = df.withColumnRenamed(v, canonical)
                break
    df = df.withColumn("EVENT_TYPE", F.lit(event_type))
    keep = [c for c in ["PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE",
             "EVENT_TYPE", "CLAIM_ID", "DATA_SOURCE"] if c in df.columns]
    return df.select(*keep)

def sanitize_column_name(name):
    import re as _re
    name = _re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    name = _re.sub(r'_+', '_', name).strip('_')
    return name.upper()

print("Standardization functions loaded.")

In [0]:
# ============================================================================
# LOAD AND STANDARDIZE LAB CLAIMS
# ============================================================================

lab_claims = spark.read.format("delta").load(lab_file_path)
lab_claims = standardize(lab_claims, "lab")

# Ensure EVENT_DATE is date type
lab_claims = lab_claims.withColumn("EVENT_DATE", F.col("EVENT_DATE").cast("date"))

# Sample 10K HCPs randomly for testing (includes both old and new claims)
if sample_hcp_count > 0:
    sampled_hcps = (lab_claims
        .select("PROVIDER_ID")
        .distinct()
        .orderBy(F.rand())
        .limit(sample_hcp_count)
    )
    lab_claims = lab_claims.join(sampled_hcps, "PROVIDER_ID", "inner")
    print(f"Sampled {sample_hcp_count} HCPs randomly")

print(f"Lab claims loaded: {lab_claims.count()} rows")
display(lab_claims.limit(10))

In [0]:
# ============================================================================
# COHORT ANCHOR DATES GENERATION (Section 2)
# ============================================================================

def generate_cohort_anchor_dates(cohort_start, cohort_end, months_incr, lookback_dur, cohort_offset=1):
    anchors = []
    current = pd.Timestamp(cohort_start)
    end = pd.Timestamp(cohort_end)
    cohort_idx = 1
    while current <= end:
        cohort_label = f"I{cohort_idx:02d}"
        if cohort_offset == 1:
            lookback_end = (current - pd.DateOffset(months=2)).to_period('M').end_time.normalize()
        else:
            month_start = current.to_period('M').start_time
            if current.day >= cohort_offset:
                custom_month_start = month_start + pd.DateOffset(days=cohort_offset - 1)
            else:
                custom_month_start = (month_start - pd.DateOffset(months=1)) + pd.DateOffset(days=cohort_offset - 1)
            lookback_end = (custom_month_start - pd.DateOffset(months=1)) - pd.DateOffset(days=1)
        lookback_start = lookback_end - pd.Timedelta(days=lookback_dur)
        anchors.append({"COHORT": cohort_label, "ANCHOR_DATE": current.strftime("%Y-%m-%d"),
                       "LOOKBACK_END_DATE": lookback_end.strftime("%Y-%m-%d"),
                       "LOOKBACK_START_DATE": lookback_start.strftime("%Y-%m-%d")})
        current = current + pd.DateOffset(months=months_incr)
        cohort_idx += 1
    return pd.DataFrame(anchors)

cohort_anchor_pd = generate_cohort_anchor_dates(cohort_start_date, cohort_end_date, months_increment, lookback_duration, cohort_day_offset)
cohort_anchor_dates = spark.createDataFrame(cohort_anchor_pd)
cohort_anchor_dates = cohort_anchor_dates \
    .withColumn("ANCHOR_DATE", F.col("ANCHOR_DATE").cast("date")) \
    .withColumn("LOOKBACK_END_DATE", F.col("LOOKBACK_END_DATE").cast("date")) \
    .withColumn("LOOKBACK_START_DATE", F.col("LOOKBACK_START_DATE").cast("date"))
cohort_anchor_pd.to_csv(anchor_dates_path, index=False)

print(f"Generated {cohort_anchor_pd.shape[0]} cohorts:")
display(cohort_anchor_dates)

---
# Feature Type 1: First Occurrence (Section 3)

## What it answers
*How many days before the anchor date did this lab test first appear in a patient's journey?*

## Column Naming
`max(EVENT_TIME)_{EVENT_NAME}_first_occurrence`

## Lab-specific: Typical event count ~2 distinct test types

In [0]:
# ============================================================================
# FIRST OCCURRENCE - Step 1: Collapse raw claims (Section 3)
# ============================================================================

first_occ_intermediate = (lab_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .groupBy("PROVIDER_ID", "EVENT_NAME")
    .agg(F.min("EVENT_DATE").alias("FIRST_OCCURRENCE_DATE"))
)

first_occ_intermediate_path = f"{output_base_path}/lab_first_occ_intermediate"
first_occ_intermediate.write.format("parquet").mode("overwrite").save(first_occ_intermediate_path)
first_occ_intermediate = spark.read.parquet(first_occ_intermediate_path)

print(f"First occurrence intermediate: {first_occ_intermediate.count()} rows")
display(first_occ_intermediate.limit(10))

In [0]:
# ============================================================================
# FIRST OCCURRENCE - Step 2: Attach cohort + Step 3: Pivot wide
# ============================================================================

with_cohort = (first_occ_intermediate
    .crossJoin(F.broadcast(cohort_anchor_dates))
    .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("FIRST_OCCURRENCE_DATE")))
    .filter(F.col("EVENT_TIME") >= 0)
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT")))
)

if event_prevalence > 0.00:
    ec = with_cohort.groupBy("EVENT_NAME").agg(F.countDistinct("PROVIDER_ID").alias("hcp_count"))
    th = with_cohort.select("PROVIDER_ID").distinct().count()
    ek = ec.filter(F.col("hcp_count") / F.lit(th) >= event_prevalence).select("EVENT_NAME").collect()
    with_cohort = with_cohort.filter(F.col("EVENT_NAME").isin([r["EVENT_NAME"] for r in ek]))

cl = [r["COHORT"] for r in cohort_anchor_dates.select("COHORT").collect()]
br = []
for i in range(0, len(cl), batch_size):
    bc = cl[i:i + batch_size]
    bdf = with_cohort.filter(F.col("COHORT").isin(bc))
    br.append(bdf.groupBy("HCP_COHORT_ID").pivot("EVENT_NAME").agg(F.max("EVENT_TIME")))

first_occurrence_features = br[0]
for b in br[1:]:
    first_occurrence_features = first_occurrence_features.unionByName(b, allowMissingColumns=True)

ic = ["HCP_COHORT_ID"]
re = [F.col(f"`{c}`").alias(f"MAX_EVENT_TIME_{sanitize_column_name(c)}_FIRST_OCCURRENCE") if c not in ic else F.col(c) for c in first_occurrence_features.columns]
first_occurrence_features = first_occurrence_features.select(*re)

first_occ_output_path = f"{output_base_path}/lab_first_occurrence"
first_occurrence_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(first_occ_output_path)

print(f"First occurrence: {first_occurrence_features.count()} rows, {len(first_occurrence_features.columns)} columns")
display(first_occurrence_features.limit(10))

---
# Feature Type 2: Recency (Section 4)

## What it answers
*How recently did lab tests occur, and how concentrated was the activity?*

## Critical: Recency uses ONLY the 12M window (NOT full history)

## Six Sub-Features (same structure as RX/DX/PX)
1. Most Recent Event: `min(EVENT_TIME)_{EVENT_NAME}_time_min`
2. Total and Unique Events: `total_events`, `unique_events`
3. Counts by Time Window: `total_events_time_{N}_days`, `unique_events_time_{N}_days`
4. Counts by Type: `total_{type}`, `unique_{type}`
5. Counts by Type x Window: `total_events_time_{N}_days_{type}`, `unique_events_time_{N}_days_{type}`
6. Skewness: `skewness_time_{N}_days_{type}`

In [0]:
# ============================================================================
# RECENCY - 12M Filter + All Sub-features (Section 4)
# ============================================================================

recency_claims = (lab_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .crossJoin(F.broadcast(cohort_anchor_dates))
)

recency_claims_12m = recency_claims.filter(
    (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)

recency_claims_12m = recency_claims_12m \
    .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("EVENT_DATE"))) \
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT"))) \
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))

# Sub-feature 1: Most Recent Event (pre-aggregation pattern)
recency_latest = (lab_claims.filter(F.col("PROVIDER_ID").isNotNull()).filter(F.col("EVENT_NAME").isNotNull())
    .groupBy("PROVIDER_ID", "EVENT_NAME").agg(F.max("EVENT_DATE").alias("LATEST_EVENT_DATE")))

recency_sub1 = (recency_latest.crossJoin(F.broadcast(cohort_anchor_dates))
    .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("LATEST_EVENT_DATE")))
    .filter(F.col("EVENT_TIME") >= 0)
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT")))
    .filter((F.col("LATEST_EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
            (F.col("LATEST_EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))))

cl = [r["COHORT"] for r in cohort_anchor_dates.select("COHORT").collect()]
bs1 = []
for i in range(0, len(cl), batch_size):
    bc = cl[i:i + batch_size]
    bdf = recency_sub1.filter(F.col("COHORT").isin(bc))
    bs1.append(bdf.groupBy("HCP_COHORT_ID").pivot("EVENT_NAME").agg(F.min("EVENT_TIME")))
r1 = bs1[0]
for b in bs1[1:]:
    r1 = r1.unionByName(b, allowMissingColumns=True)
ic = ["HCP_COHORT_ID"]
re1 = [F.col(f"`{c}`").alias(f"MIN_EVENT_TIME_{sanitize_column_name(c)}_TIME_MIN") if c not in ic else F.col(c) for c in r1.columns]
recency_sub1_result = r1.select(*re1)

# Sub-feature 2: Total and Unique Events
recency_sub2 = recency_claims_12m.groupBy("HCP_COHORT_ID").agg(
    F.count("*").alias("total_events"), F.countDistinct("EVENT_NAME").alias("unique_events"))

# Sub-feature 3: Counts by Time Window
recency_sub3 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for wn in time_windows_recency:
    wc = recency_claims_12m.filter(F.col("EVENT_TIME") <= wn).groupBy("HCP_COHORT_ID").agg(
        F.count("*").alias(f"total_events_time_{wn}_days"),
        F.countDistinct("EVENT_NAME").alias(f"unique_events_time_{wn}_days"))
    recency_sub3 = recency_sub3.join(wc, on="HCP_COHORT_ID", how="left")
for wn in time_windows_recency:
    recency_sub3 = recency_sub3.fillna(0, subset=[f"total_events_time_{wn}_days", f"unique_events_time_{wn}_days"])

# Sub-feature 4: Counts by Type
event_types = [r["EVENT_TYPE"] for r in recency_claims_12m.select("EVENT_TYPE").distinct().collect()]
recency_sub4 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for et in event_types:
    tc = recency_claims_12m.filter(F.col("EVENT_TYPE") == et).groupBy("HCP_COHORT_ID").agg(
        F.count("*").alias(f"total_{et}"), F.countDistinct("EVENT_NAME").alias(f"unique_{et}"))
    recency_sub4 = recency_sub4.join(tc, on="HCP_COHORT_ID", how="left")
for et in event_types:
    recency_sub4 = recency_sub4.fillna(0, subset=[f"total_{et}", f"unique_{et}"])

# Sub-feature 5: Counts by Type x Window
recency_sub5 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for et in event_types:
    for wn in time_windows_recency:
        twc = recency_claims_12m.filter((F.col("EVENT_TYPE") == et) & (F.col("EVENT_TIME") <= wn)).groupBy("HCP_COHORT_ID").agg(
            F.count("*").alias(f"total_events_time_{wn}_days_{et}"),
            F.countDistinct("EVENT_NAME").alias(f"unique_events_time_{wn}_days_{et}"))
        recency_sub5 = recency_sub5.join(twc, on="HCP_COHORT_ID", how="left")
for et in event_types:
    for wn in time_windows_recency:
        recency_sub5 = recency_sub5.fillna(0, subset=[f"total_events_time_{wn}_days_{et}", f"unique_events_time_{wn}_days_{et}"])

# Sub-feature 6: Skewness
recency_sub6 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for et in event_types:
    for wn in time_windows_recency:
        sk = recency_claims_12m.filter((F.col("EVENT_TYPE") == et) & (F.col("EVENT_TIME") <= wn)).groupBy("HCP_COHORT_ID").agg(
            F.skewness("EVENT_TIME").alias(f"skewness_time_{wn}_days_{et}"))
        recency_sub6 = recency_sub6.join(sk, on="HCP_COHORT_ID", how="left")

# Combine all
recency_combined = (recency_sub1_result
    .join(recency_sub2, on="HCP_COHORT_ID", how="left")
    .join(recency_sub3, on="HCP_COHORT_ID", how="left")
    .join(recency_sub4, on="HCP_COHORT_ID", how="left")
    .join(recency_sub5, on="HCP_COHORT_ID", how="left")
    .join(recency_sub6, on="HCP_COHORT_ID", how="left")
)
recency_combined = recency_combined.withColumn("COHORT", F.substring(F.col("HCP_COHORT_ID"), -3, 3))
recency_output_path = f"{output_base_path}/lab_recency"
recency_combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("COHORT").save(recency_output_path)

print(f"Recency combined: {recency_combined.count()} rows, {len(recency_combined.columns)} columns")
display(recency_combined.limit(10))

---
# Feature Type 3: Frequency (Section 5)

## Lab-specific column naming for Sub-D
`NUM_OF_PATIENTS_THE_HCP_DID_LAB_TEST_{EVENT}_IN_LAST_{N}_DAYS`

In [0]:
# ============================================================================
# FREQUENCY - Sub-type A: Monthly Claim Count (Section 5)
# ============================================================================

freq_claims = lab_claims \
    .filter(F.col("PROVIDER_ID").isNotNull()) \
    .filter(F.col("EVENT_NAME").isNotNull()) \
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))

freq_subA = (freq_claims.groupBy("PROVIDER_ID", "SVC_DT_M").pivot("EVENT_NAME").agg(F.count(F.lit(1))))

id_a = ["PROVIDER_ID", "SVC_DT_M"]
ra = [F.col(f"`{c}`").alias(f"FREQ_{sanitize_column_name(c)}") if c not in id_a else F.col(c) for c in freq_subA.columns]
freq_subA = freq_subA.select(*ra)

freq_subA_path = f"{output_base_path}/lab_freq_subA_monthly_count"
freq_subA.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("SVC_DT_M").save(freq_subA_path)

print(f"Freq Sub-A: {freq_subA.count()} rows, {len(freq_subA.columns)} columns")
display(freq_subA.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type B: Lag Frequency (Section 5)
# ============================================================================

def fill_missing(df, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    all_hcps = df.select(group_col).distinct()
    min_d = df.select(F.min(date_col)).collect()[0][0]
    max_d = df.select(F.max(date_col)).collect()[0][0]
    ms = spark.createDataFrame(pd.date_range(start=min_d, end=max_d, freq='MS'),
        schema=StructType([StructField(date_col, DateType(), True)]))
    spine = all_hcps.crossJoin(ms)
    result = spine.join(df, on=[group_col, date_col], how="left")
    fc = [c for c in result.columns if c not in [group_col, date_col]]
    return result.fillna(0, subset=fc)

def apply_lag(df, windows, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    w = Window.partitionBy(group_col).orderBy(date_col)
    freq_cols = [c for c in df.columns if c not in [group_col, date_col]]
    new_cols = {}
    for c in freq_cols:
        for n in windows:
            new_cols[f"{c}_IN_LAST_{n}_MONTH"] = F.sum(c).over(w.rowsBetween(-n + 1, 0))
    return df.withColumns(new_cols)

def process_and_join(df, windows):
    filled = fill_missing(df)
    lagged = apply_lag(filled, windows)
    kc = ["PROVIDER_ID", "SVC_DT_M"]
    keep = kc + [c for c in lagged.columns if "_IN_LAST_" in c]
    result = lagged.select(*keep)
    return result

freq_subB = process_and_join(freq_subA, lag_windows_freq)
freq_subB_path = f"{output_base_path}/lab_freq_subB_lag"
freq_subB.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("SVC_DT_M").save(freq_subB_path)

print(f"Freq Sub-B: {freq_subB.count()} rows, {len(freq_subB.columns)} columns")
display(freq_subB.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type C: Change in Frequency (Section 5)
# ============================================================================

event_bases = set()
for c in [c for c in freq_subB.columns if "_IN_LAST_" in c and c not in ["PROVIDER_ID", "SVC_DT_M"]]:
    event_bases.add(c.split("_IN_LAST_")[0])

new_cols = {}
for eb in event_bases:
    c12 = f"{eb}_IN_LAST_12_MONTH"
    if c12 not in freq_subB.columns:
        continue
    for n in change_windows_freq:
        cn = f"{eb}_IN_LAST_{n}_MONTH"
        if cn not in freq_subB.columns:
            continue
        cc = f"CHANGE_IN_FREQ_OF_HCP_CLAIMS_FOR_{eb.replace('FREQ_', '')}_IN_LAST_{n}_MONTH"
        new_cols[cc] = (F.col(cn) / F.lit(n)) - ((F.col(c12) - F.col(cn)) / F.lit(12 - n))

freq_subC = freq_subB.withColumns(new_cols)

kc_c = ["PROVIDER_ID", "SVC_DT_M"]
freq_subC = freq_subC.select(*kc_c, *[c for c in freq_subC.columns if c.startswith("CHANGE_IN_FREQ_")])

freq_subC_path = f"{output_base_path}/lab_freq_subC_change"
freq_subC.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("SVC_DT_M").save(freq_subC_path)

print(f"Freq Sub-C: {freq_subC.count()} rows, {len(freq_subC.columns)} columns")
display(freq_subC.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type D: Unique Patient Counts per HCP (Section 5)
# ============================================================================
# Lab column naming: NUM_OF_PATIENTS_THE_HCP_DID_LAB_TEST_{EVENT}_IN_LAST_{N}_DAYS
# Grain: [PROVIDER_ID, COHORT]

freq_subD_claims = (lab_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .filter(F.col("PATIENT_ID").isNotNull())
    .crossJoin(F.broadcast(cohort_anchor_dates))
)
freq_subD_claims = freq_subD_claims.filter(
    (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)
freq_subD_claims = freq_subD_claims.withColumn("days_diff", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("EVENT_DATE")))

for wn in patient_count_windows_freq:
    freq_subD_claims = freq_subD_claims.withColumn(f"within_{wn}_days", F.when(F.col("days_diff") <= wn, F.lit(1)).otherwise(F.lit(0)))

freq_subD_claims = freq_subD_claims.dropDuplicates(["PROVIDER_ID", "COHORT", "EVENT_NAME", "PATIENT_ID"])

freq_subD_results = []
for wn in patient_count_windows_freq:
    wdf = freq_subD_claims.filter(F.col(f"within_{wn}_days") == 1)
    counts = wdf.groupBy("PROVIDER_ID", "COHORT", "EVENT_NAME").agg(F.countDistinct("PATIENT_ID").alias("pc"))
    piv = counts.groupBy("PROVIDER_ID", "COHORT").pivot("EVENT_NAME").agg(F.sum("pc"))
    id_d = ["PROVIDER_ID", "COHORT"]
    rd = [F.col(f"`{c}`").alias(f"NUM_OF_PATIENTS_THE_HCP_DID_LAB_TEST_{sanitize_column_name(c)}_IN_LAST_{wn}_DAYS") if c not in id_d else F.col(c) for c in piv.columns]
    freq_subD_results.append(piv.select(*rd))

freq_subD = freq_subD_results[0]
for r in freq_subD_results[1:]:
    freq_subD = freq_subD.join(r, on=["PROVIDER_ID", "COHORT"], how="left")

freq_subD_path = f"{output_base_path}/lab_freq_subD_patient_counts"
freq_subD.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("COHORT").save(freq_subD_path)

print(f"Freq Sub-D: {freq_subD.count()} rows, {len(freq_subD.columns)} columns")
display(freq_subD.limit(10))

---
# Feature Type 4: Hospitalization (Section 6)

## NOT APPLICABLE FOR LAB

Per the skill specification (Section 6):
> **DX and PX only** — no RX or Lab hospitalization features exist.

Hospitalization features require `DATA_SOURCE = 'HX'` (hospitalized claims) and are only computed for Diagnosis (DX) and Procedure (PX) claim types. Lab test claims do not have a hospitalization dimension.

In [0]:
# ============================================================================
# LAB PATIENT FEATURES - FINAL OUTPUT SUMMARY (Section 8)
# ============================================================================

print("=" * 70)
print("LAB PATIENT FEATURES - GENERATION COMPLETE")
print("=" * 70)
print()
print("Feature outputs generated:")
print()
print("1. First Occurrence (HCP_COHORT_ID grain)")
print(f"   Path: {output_base_path}/lab_first_occurrence")
print()
print("2. Recency (HCP_COHORT_ID grain, 6 sub-features)")
print(f"   Path: {output_base_path}/lab_recency")
print()
print("3. Frequency:")
print(f"   Sub-A (Monthly): {output_base_path}/lab_freq_subA_monthly_count")
print(f"   Sub-B (Lag): {output_base_path}/lab_freq_subB_lag")
print(f"   Sub-C (Change): {output_base_path}/lab_freq_subC_change")
print(f"   Sub-D (Patients): {output_base_path}/lab_freq_subD_patient_counts")
print()
print("4. Hospitalization: NOT APPLICABLE FOR LAB")
print()
print("=" * 70)
print("Join strategy (Section 8.2):")
print("  Step 1: Build skeleton from hcp_universe x cohort_anchor_dates")
print("  Step 2: LEFT JOIN first_occurrence ON HCP_COHORT_ID")
print("  Step 3: LEFT JOIN recency ON HCP_COHORT_ID")
print("  Step 4: LEFT JOIN freq_subABC ON [PROVIDER_ID, SVC_DT_M]")
print("  Step 5: LEFT JOIN freq_subD ON [PROVIDER_ID, COHORT]")
print("  Step 6: Lab features: _LAB suffix on non-key columns (to avoid collisions)")
print("=" * 70)